# 2. データを分析する

どんな属性の人が生き残ったのかを、学習用データから調べる。

ここで「効きそうな列」に当たりを付けておくと、次の前処理で
どの列を残し、どの列に手をかけるかを決められる。

In [ ]:
import pandas as pd

# 分析に使うのは学習用データだけ。test には survived が無いので、
# 「どんな人が生き残ったか」は train からしか分からない
train = pd.read_csv("../data/raw/train.csv", index_col=0)

## 1. そもそも何人が生き残ったのか

予測の対象である `survived` に、0 と 1 がそれぞれ何件あるかを数える。

In [ ]:
# .value_counts() は、その列に出てくる値ごとの個数を数える
#   カテゴリや 0/1 の列の内訳をつかむときの定番
train["survived"].value_counts()

# 出力の見方
#   0 が 266 人（助からなかった）、1 が 179 人（生還した）
#   Name: count は「数えた個数の列ですよ」という pandas 側のラベル。データの一部ではない

In [ ]:
# 割合で見たいときは normalize=True。個数ではなく全体に占める比率を返す
train["survived"].value_counts(normalize=True)

# 生還率は約 40%。映画の印象より高い
# この 40% が後で効いてくる。「全員が死んだ」と答えるだけのモデルでも 60% 当たってしまうので、
# 精度 60% は「何も学習できていない」のと同じ。これを基準（ベースライン）と呼ぶ

## 2. 数値の列をまとめて眺める

`.describe()` で、数値の列の代表的な統計量を一度に出す。
外れ値や、値の散らばり方の見当を付けるのに使う。

In [ ]:
# .describe() は数値列だけを対象に、件数・平均・ばらつき・最小最大などを表にする
#   文字列の列（sex, embarked）は自動で除かれる
train.describe()

# 出力の見方（縦に並ぶ行の意味）
#   count  欠損していない件数。age だけ 360 と少ないのは欠損があるため
#   mean   平均値
#   std    標準偏差。値がどれくらいばらついているか。大きいほど散らばっている
#   min    最小値
#   25%    小さい順に並べて 1/4 番目の値
#   50%    真ん中の値（中央値）
#   75%    小さい順に並べて 3/4 番目の値
#   max    最大値

### 読み取れること

**`pclass` の 50%（中央値）が 3** — 乗客の半数以上が一番下の3等客室にいた。

**`age` の平均は約 29 歳** — 若い乗客が多い。最小は 0.67（生後8か月）、最大は 80 歳。

**`fare` は平均 34 に対して最大 512** — 平均の 15 倍を払った人がいる。
`std`（ばらつき）が 52 と平均より大きいことからも、一部の高額運賃が全体を引っ張っていると分かる。
こういう「極端に大きい値が少数ある」分布では、平均値はその少数に引きずられるので、
代表値としては中央値（15.0）の方が実態に近い。

**`survived` の平均が 0.402** — 0 と 1 しかない列の平均は、そのまま 1 の割合になる。
つまり生還率 40.2%。この性質は後で属性別の生存率を出すときに使う。

### 平均・中央値・最頻値

`describe()` に出てくる代表値には、それぞれ英語の名前がある。
この先ずっと出てくるので、ここで対応を押さえておく。

| 日本語 | 英語 | pandas | 意味 |
| --- | --- | --- | --- |
| 平均値 | **mean** | `.mean()` | 全部足して人数で割る |
| 中央値 | **median** | `.median()` | 順に並べてちょうど真ん中の人の値。`describe()` の `50%` がこれ |
| 最頻値 | **mode** | `.mode()` | 一番多く出てくる値 |

**平均と中央値は、ずれることがある。**

| 列 | 平均 | 中央値 | 平均より下の人 |
| --- | --- | --- | --- |
| `age` | 29.21 | 28.00 | 53.3% |
| `fare` | 33.96 | **15.00** | **75.1%** |

`age` はほぼ一致しているが、`fare` は倍以上ひらく。
512 を払った人のような極端な高額客が平均を押し上げるためで、
結果として乗客の4人に3人が「平均以下」になってしまう。

こうなると平均は代表値として役に立たない。年収の話でいつも揉めるのと同じ構図で、
一部の高額側に引っ張られる列では中央値の方が実感に近い。

最頻値は、`embarked` のように**大小関係がない値**でも使えるのが利点。
港に平均も真ん中もないが、「一番多いのはどれか」なら答えられる。

## 3. どの列が生死と関係していそうか

列を1つずつ調べる前に、`survived` との関係が強い列を **相関係数** でざっと絞り込む。

相関係数は2つの列の連動具合を -1 〜 1 で表した数値。

- **1 に近い** — 片方が大きいとき、もう片方も大きい（正の相関）
- **-1 に近い** — 片方が大きいとき、もう片方は小さい（負の相関）
- **0 に近い** — 関係が薄い

ただし相関係数は数値どうしでしか計算できない。`sex` は `male` / `female` という文字列なので
このままでは扱えず、先に数値へ変換する必要がある。

In [ ]:
# get_dummies() は文字列のカテゴリを列に分解する（ダミー化・One-Hot エンコーディング）
#   文字列の列すべてが対象なので、sex と embarked の2列がまとめて分解される
#     sex      → sex_female / sex_male                （2種類なので2列）
#     embarked → embarked_C / embarked_Q / embarked_S  （3種類なので3列）
#   元の値に当てはまる列だけが True になり、残りは False になる
#
#   変換前              変換後
#   sex     embarked    sex_female  sex_male  embarked_C  embarked_Q  embarked_S
#   female  S      →    True        False     False       False       True
#   male    C      →    False       True      True        False       False
pd.get_dummies(train).head()

In [ ]:
# .corrwith() は、表の各列と、指定した1つの列との相関係数をまとめて出す
pd.get_dummies(train).corrwith(train["survived"])

# 出力の見方
#   survived が 1.000 なのは自分自身との相関なので、当然そうなる。判断には使わない

### 読み取れること

| 列 | 相関係数 | 意味 |
| --- | --- | --- |
| `sex_female` | **+0.559** | 女性であるほど生還している。最も強い |
| `sex_male` | **-0.559** | 男性であるほど助かっていない |
| `pclass` | **-0.358** | 数字が大きい（＝下の等級）ほど助かっていない |
| `fare` | +0.259 | 運賃が高いほど生還している |
| `embarked_C` | +0.183 | シェルブール港から乗った人はやや生還が多い |
| `embarked_S` | -0.173 | サウサンプトン港から乗った人はやや少ない |
| `age` | -0.081 | ほとんど関係が見えない |
| `sibsp` | -0.045 | ほとんど関係が見えない |
| `parch` | +0.080 | ほとんど関係が見えない |

**効いているのは性別と客室クラス。** 次点で運賃だが、良い客室ほど運賃が高いので、
`pclass` と同じことを別の角度から見ているだけの可能性が高い。

読むときの注意が3つある。

**1. `sex_female` と `sex_male` は符号が逆なだけの同じ情報**

男でなければ女なので、片方が決まればもう片方も決まる。
2列に分けた意味はなく、実際の学習では片方だけ使えば足りる。

**2. `pclass` の符号がマイナスなのは「等級が低いほど死んだ」という意味**

`pclass` は 1 が最上級で 3 が最下級と、数字の大小と豪華さが逆向きに対応している。
だから「`pclass` の数字が大きいほど `survived` が小さい」＝「下の等級ほど助からなかった」となる。
相関係数の符号は、値の並び方の向きに左右されるので、符号だけ見て判断しない。

**3. `age` の相関が弱くても「年齢は関係ない」とは言い切れない**

相関係数が捉えるのは直線的な関係だけ。
「子どもと高齢者は助かり、働き盛りは助からない」のような山なりの関係は 0 に近く出てしまう。
実際タイタニックでは子どもが優先されたと言われており、年齢は区切り方を変えれば効く可能性がある。

## 4. 実際の生存率を属性ごとに確かめる

相関係数はあくまで目安なので、効いていそうな列について実際の生存率を出して裏を取る。

`survived` は 0 と 1 なので、**グループごとの平均値がそのグループの生存率**になる。

In [ ]:
# .groupby("列名") でその列の値ごとにグループ分けし、.mean() で各グループの平均を出す
#   ["survived"] で対象の列を絞ってから平均を取る
train.groupby("pclass")["survived"].mean()

# 出力の見方
#   1（1等客室）: 0.685 → 68.5% が生還
#   2（2等客室）: 0.443 → 44.3%
#   3（3等客室）: 0.258 → 25.8%

In [ ]:
train.groupby("sex")["survived"].mean()

# female: 0.776 → 77.6% が生還
# male  : 0.201 → 20.1%

In [ ]:
# 生存率だけ見ると人数を見落とすので、.agg() で複数の集計を同時に出しておく
#   count は人数。「9割生還」でも 2 人中 2 人なら、たまたまかもしれない
train.groupby("sex")["survived"].agg(["mean", "count"])

# female 156 人 / male 289 人。どちらも十分な人数があるので、この差は偶然ではなさそう

In [ ]:
# 性別と客室クラスを掛け合わせて見る。両方を同時に指定するとグループが細かくなる
train.groupby(["sex", "pclass"])["survived"].agg(["mean", "count"])

# 3等客室の女性（58.5%）は、1等客室の男性（43.6%）より生存率が高い。
# 客室クラスより性別の影響の方が大きかったことが、ここからも読み取れる
#
# 女性は 1等 94.3% → 2等 86.8% → 3等 58.5%
# 男性は 1等 43.6% → 2等 16.9% → 3等 13.7%
# どの等級でも女性が男性を上回っている

## この章のまとめ

- 学習用データ 445 人のうち生還は 179 人、**生存率は 40.2%**。
  何も学習していないモデルでも 60% 当たるので、精度はこれを上回って初めて意味がある
- **性別が最も効く。** 女性 77.6% に対して男性 20.1%
- **次に客室クラス。** 1等 68.5% → 2等 44.3% → 3等 25.8%
- 性別と客室クラスを掛け合わせると、**3等の女性（58.5%）が1等の男性（43.6%）を上回る**。
  どの等級でも女性が男性より高く、性別の影響の方が強い
- `age` は相関係数上は弱いが、区切り方次第で効く可能性が残っている